In [3]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.metrics import confusion_matrix, cohen_kappa_score, accuracy_score, classification_report
from PIL import Image
import timm

# --- CONFIG ---
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
BASE_PATH = '/kaggle/input/datasets/mariaherrerot/aptos2019/'
TRAIN_IMG_DIR = os.path.join(BASE_PATH, 'train_images/train_images')
VAL_IMG_DIR = os.path.join(BASE_PATH, 'val_images/val_images')

# Resolution 320 for the CNN path to catch tiny lesions
IMG_SIZE = 320 
BATCH_SIZE = 16 
LR = 4e-5

class AptosFinalDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(os.path.join(BASE_PATH, csv_file))
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f"{self.df.iloc[idx, 0]}.png")
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        label = self.df.iloc[idx, 1]
        return image, torch.tensor(label, dtype=torch.long)

# --- NOVELTY: MULTI-RESOLUTION HETEROGENEOUS ENSEMBLE ---
class ResearchEnsemble(nn.Module):
    def __init__(self):
        super().__init__()
        # ConvNeXt (Structural Expert) - Processes 320x320
        self.cnn = timm.create_model('convnext_tiny', pretrained=True, num_classes=5)
        # Swin (Context Expert) - Processes 224x224
        self.vit = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=5)
        
    def forward(self, x):
        # Path 1: CNN sees high-res (320px)
        out_cnn = self.cnn(x)
        
        # Path 2: ViT sees downsampled res (224px) to avoid AssertionError
        x_vit = F.interpolate(x, size=(224, 224), mode='bicubic', align_corners=False)
        out_vit = self.vit(x_vit)
        
        # Weighted Fusion
        return (0.6 * out_cnn) + (0.4 * out_vit)

def run_final_push():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ResearchEnsemble().to(device)
    
    # Class weights to prioritize Severe (Stage 3) and Proliferative (Stage 4)
    weights = torch.tensor([1.0, 1.8, 1.5, 4.5, 3.0]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    train_loader = DataLoader(AptosFinalDataset('train_1.csv', TRAIN_IMG_DIR, train_tf), 
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    val_loader = DataLoader(AptosFinalDataset('valid.csv', VAL_IMG_DIR, train_tf), 
                            batch_size=BATCH_SIZE, num_workers=4)

    print("🚀 Final Resolution-Aware Ensemble training initiated...")
    best_acc = 0
    
    for epoch in range(12):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out = model(imgs)
                preds.extend(torch.argmax(out, 1).cpu().numpy())
                targets.extend(labels.cpu().numpy())

        acc = accuracy_score(targets, preds)
        print(f"Epoch {epoch+1:02d} | Accuracy: {acc:.4%} | Kappa: {cohen_kappa_score(targets, preds, weights='quadratic'):.4f}")

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), 'multi_res_ensemble.pth')

    # --- REPORTING ---
    model.load_state_dict(torch.load('multi_res_ensemble.pth'))
    model.eval()
    f_preds, f_targets = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            f_preds.extend(torch.argmax(model(imgs), 1).cpu().numpy())
            f_targets.extend(labels.cpu().numpy())

    print("\n" + "="*50)
    print(f"OVERALL ACCURACY: {accuracy_score(f_targets, f_preds):.4%}")
    print(f"OVERALL KAPPA:    {cohen_kappa_score(f_targets, f_preds, weights='quadratic'):.4f}")
    print("-" * 50)
    print(classification_report(f_targets, f_preds, target_names=['No DR', 'Mild', 'Moderate', 'Severe', 'Prolif']))

if __name__ == "__main__":
    run_final_push()

🚀 Final Resolution-Aware Ensemble training initiated...
Epoch 01 | Accuracy: 70.7650% | Kappa: 0.7891
Epoch 02 | Accuracy: 81.1475% | Kappa: 0.8874
Epoch 03 | Accuracy: 83.6066% | Kappa: 0.9125
Epoch 04 | Accuracy: 80.0546% | Kappa: 0.9019
Epoch 05 | Accuracy: 84.1530% | Kappa: 0.9097
Epoch 06 | Accuracy: 84.9727% | Kappa: 0.9064
Epoch 07 | Accuracy: 82.7869% | Kappa: 0.8799
Epoch 08 | Accuracy: 78.4153% | Kappa: 0.8682
Epoch 09 | Accuracy: 80.3279% | Kappa: 0.8754
Epoch 10 | Accuracy: 85.5191% | Kappa: 0.9034
Epoch 11 | Accuracy: 83.0601% | Kappa: 0.8862
Epoch 12 | Accuracy: 84.9727% | Kappa: 0.9110

OVERALL ACCURACY: 84.4262%
OVERALL KAPPA:    0.9027
--------------------------------------------------
              precision    recall  f1-score   support

       No DR       0.99      0.99      0.99       172
        Mild       0.68      0.68      0.68        40
    Moderate       0.77      0.83      0.80       104
      Severe       0.62      0.36      0.46        22
      Prolif     